In [1]:
import pybamm
import numpy as np


v_nom = 3.63

n_c = []
n_c_at_vnom = []
c_rates_charge = np.arange(0.1, 0.8, 0.1) 
solutions_charge = []
for i in c_rates_charge:
    experiment = pybamm.Experiment(
        [
            (
                f"Discharge at {i}C until 2.5V"
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=1)
    solutions_charge.append(sol)
    new_n_c = 1 - sol["Current [A]"].data * (sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_c.append(new_n_c)
    n_c_at_vnom.append(new_n_c[np.argmin(np.abs(sol["Terminal voltage [V]"].data - 3.63))])
n_d = []
n_d_at_vnom = []
c_rates_discharge = np.arange(0.2, 3.2, 0.4)
solutions_discharge = []
for i in c_rates_discharge:
    experiment = pybamm.Experiment(
        [
            (
                f"Charge at {i}C until 4.2V",
            )
        ]
    )
    model = pybamm.lithium_ion.SPM()
    sim = pybamm.Simulation(model, experiment=experiment, parameter_values=pybamm.ParameterValues("Chen2020"))
    sol = sim.solve(initial_soc=0)
    solutions_discharge.append(sol)
    new_n_d = 1 - sol["Current [A]"].data * (-sol["Local ECM resistance [Ohm]"].data) / sol["Terminal voltage [V]"].data
    n_d.append(new_n_d)
    n_d_at_vnom.append(new_n_d[np.argmin(np.abs(sol["Terminal voltage [V]"].data - 3.63))])

2024-05-15 15:54:11.802 - [WARNING] citations.read_citations(76): Citations could not be read because the 'pybtex' library is not installed. Install 'pybamm[cite]' to enable citation reading.


In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03)
fig.update_layout(
    width=800,
    height=600,
    margin={"r": 0, "l": 0, "t": 0.1, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=13,
    legend2={"y": 0.5, "yanchor": "top"},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
for i in range(len(n_c)):
    fig.add_trace(
        go.Scatter(x=solutions_charge[i]["Terminal voltage [V]"].data[1:], y=n_c[i][1:], name=f"{c_rates_charge[i]:.1f}C", legend="legend", mode="lines"), row=1, col=1 
    )
fig.add_vline(x=v_nom)
fig.add_trace(go.Scatter(x=[v_nom], y=[np.mean(n_c_at_vnom)], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=1, col=1)
fig.update_yaxes(row=1, col=1, title_text="Charging efficiency", range=[0.85, 1.001], tickmode = 'linear', tick0 = 0.86, dtick = 0.02)
for i in range(len(n_d)):
    fig.add_trace(
        go.Scatter(x=solutions_discharge[i]["Terminal voltage [V]"].data[1:], y=n_d[i][1:], name=f"-{c_rates_discharge[i]:.1f}C", legend="legend2", mode="lines"), row=2, col=1 
    )
fig.add_vline(x=v_nom)
fig.add_trace(go.Scatter(x=[v_nom], y=[np.mean(n_d_at_vnom)], marker_symbol="x", marker_color = "red", marker_size=15, showlegend=False), row=2, col=1)
fig.update_xaxes(row=2, col=1, title="Terminal Voltage", ticksuffix="V")
fig.update_yaxes(row=2, col=1, title_text="Discharging efficiency", range=[1, 1.12])
fig.write_image("../thesis/Images/charging_efficiency.pdf", "pdf")